# 01 — Load & Inspect Dataset

Loads the fire & smoke dataset from Google Drive, inspects its structure, and visualizes key statistics.

| | |
|---|---|
| **Dataset** | `fire-smoke-new` |
| **Classes** | `smoke` (0), `fire` (1) |
| **Source** | `/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/dataset/` |

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE_TAR    = '/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/dataset/dataset.tar.gz'
LOCAL_DIR    = '/content/dataset'
DATASET_ROOT = '/content/dataset/fire-smoke-new'

if os.path.exists(DATASET_ROOT):
    print('Dataset already on VM — skipping copy.')
else:
    os.makedirs(LOCAL_DIR, exist_ok=True)
    print('Copying archive from Drive to VM...')
    shutil.copy2(DRIVE_TAR, '/content/dataset_raw')
    print('Extracting...')
    !tar -xf /content/dataset_raw -C {LOCAL_DIR}
    print('Done!')

print(f'\nDataset root: {DATASET_ROOT}')

In [ ]:
import glob

CLASS_NAMES = {0: 'smoke', 1: 'fire'}

# Count images and labels
splits = {}
for split in ['train', 'val']:
    imgs   = glob.glob(f'{DATASET_ROOT}/images/{split}/*.jpg') + \
             glob.glob(f'{DATASET_ROOT}/images/{split}/*.png')
    labels = glob.glob(f'{DATASET_ROOT}/labels/{split}/*.txt')
    splits[split] = {'images': imgs, 'labels': labels}

print('=' * 40)
print('  Dataset Summary')
print('=' * 40)
for split, data in splits.items():
    print(f'  {split:6s} → {len(data["images"]):6d} images  |  {len(data["labels"]):6d} labels')
print(f'  Total  → {sum(len(d["images"]) for d in splits.values()):6d} images')
print('=' * 40)
print(f'  Classes: {list(CLASS_NAMES.values())}')
print('=' * 40)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Count bounding boxes per class across all splits
all_counts = {split: Counter() for split in splits}

for split, data in splits.items():
    for lf in data['labels']:
        with open(lf) as fh:
            for line in fh:
                parts = line.strip().split()
                if parts:
                    all_counts[split][int(parts[0])] += 1

# Print summary
print('Bounding box counts per class:')
for split, counter in all_counts.items():
    print(f'\n  [{split}]')
    for cls_id, name in CLASS_NAMES.items():
        print(f'    {name:6s} (id={cls_id}): {counter[cls_id]:7,} boxes')

# Bar chart — side by side for train vs val
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#5599ff', '#ff5555']

for ax, (split, counter) in zip(axes, all_counts.items()):
    names  = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]
    counts = [counter[i] for i in sorted(CLASS_NAMES)]
    bars = ax.bar(names, counts, color=colors)
    ax.bar_label(bars, fmt='%,.0f')
    ax.set_title(f'Class distribution — {split}')
    ax.set_ylabel('Number of bounding boxes')
    ax.set_ylim(0, max(counts) * 1.15)

plt.suptitle('Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

colors = ['#5599ff', '#ff5555']

# Analyse bounding box sizes (width, height in relative YOLO coords)
bw_all, bh_all, cls_all = [], [], []

for lf in splits['train']['labels']:
    with open(lf) as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_all.append(int(parts[0]))
                bw_all.append(float(parts[3]))
                bh_all.append(float(parts[4]))

bw_all = np.array(bw_all)
bh_all = np.array(bh_all)

print('Bounding box size stats (relative, 0–1):')
print(f'  Width  — mean: {bw_all.mean():.3f}  median: {np.median(bw_all):.3f}  max: {bw_all.max():.3f}')
print(f'  Height — mean: {bh_all.mean():.3f}  median: {np.median(bh_all):.3f}  max: {bh_all.max():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: box width vs height coloured by class
for cls_id, name in CLASS_NAMES.items():
    mask = np.array(cls_all) == cls_id
    axes[0].scatter(bw_all[mask], bh_all[mask], alpha=0.1, s=2,
                    label=name, color=colors[cls_id])
axes[0].set_xlabel('Box width (relative)')
axes[0].set_ylabel('Box height (relative)')
axes[0].set_title('Bounding box size distribution')
axes[0].legend(markerscale=5)

# Histogram of box areas
areas = bw_all * bh_all
axes[1].hist(areas, bins=50, color='#aaaaff', edgecolor='white')
axes[1].set_xlabel('Box area (relative)')
axes[1].set_ylabel('Count')
axes[1].set_title('Bounding box area histogram (train)')

plt.tight_layout()
plt.show()

In [ ]:
import random, cv2

CLASS_COLORS = {
    0: (85, 153, 255),   # smoke — blue
    1: (255, 85,  85),   # fire  — red
}

def draw_boxes(img_path, label_path):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path) as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1, y1 = int((cx - bw/2)*w), int((cy - bh/2)*h)
                x2, y2 = int((cx + bw/2)*w), int((cy + bh/2)*h)
                color = CLASS_COLORS.get(cls, (255, 255, 0))
                cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                label = CLASS_NAMES.get(cls, str(cls))
                cv2.putText(img, label, (x1, max(y1-6, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

# Sample 9 images from train
sample = random.sample(splits['train']['images'], min(9, len(splits['train']['images'])))

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, img_path in zip(axes.flatten(), sample):
    label_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    ax.imshow(draw_boxes(img_path, label_path))
    ax.set_title(os.path.basename(img_path)[:40], fontsize=7)
    ax.axis('off')

plt.suptitle('Sample Training Images with Ground-Truth Boxes\n(blue=smoke, red=fire)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✅ Done!

Dataset looks good. Head to the next notebook to start training.

➡️ Next: **02 — Train Model**